In [1]:
import pandas as pd
import sqlite3

# Wczytanie oczyszczonych danych
df = pd.read_csv('procurement_clean.csv')

# Szybka weryfikacja, że to na pewno ten plik (powinno być 776 wierszy, 19 kolumn)
print("Kształt danych:", df.shape)
print("\nKolumny:")
print(df.columns.tolist())
df.head()

Kształt danych: (776, 21)

Kolumny:
['PO_ID', 'Supplier', 'Order_Date', 'Delivery_Date', 'Item_Category', 'Order_Status', 'Quantity', 'Unit_Price', 'Negotiated_Price', 'Defective_Units', 'Compliance', 'flag_invalid_date_order', 'flag_delivered_no_date', 'flag_nondelivered_has_date', 'Lead_Time_Days', 'Order_Value', 'Savings_Amount', 'Savings_Rate_%', 'Defect_Rate_%', 'Category_Lead_Time_Benchmark', 'Is_On_Time']


,PO_ID,Supplier,Order_Date,Delivery_Date,Item_Category,Order_Status,Quantity,Unit_Price,Negotiated_Price,Defective_Units,...,flag_invalid_date_order,flag_delivered_no_date,flag_nondelivered_has_date,Lead_Time_Days,Order_Value,Savings_Amount,Savings_Rate_%,Defect_Rate_%,Category_Lead_Time_Benchmark,Is_On_Time
0,PO-00001,Alpha_Inc,2023-10-17,2023-10-25,Office Supplies,Cancelled,1176,20.13,17.81,NaN,...,False,False,True,8.0,20944.56,2728.32,11.53,NaN,10.0,1.0
1,PO-00002,Delta_Logistics,2022-04-25,2022-05-05,Office Supplies,Delivered,1509,39.32,37.34,235.0,...,False,False,False,10.0,56346.06,2987.82,5.04,15.57,10.0,1.0
2,PO-00003,Gamma_Co,2022-01-26,2022-02-15,MRO,Delivered,910,95.51,92.26,41.0,...,False,False,False,20.0,83956.60,2957.50,3.40,4.51,12.0,0.0
3,PO-00004,Beta_Supplies,2022-10-09,2022-10-28,Packaging,Delivered,1344,99.85,95.52,112.0,...,False,False,False,19.0,128378.88,5819.52,4.34,8.33,11.0,0.0
4,PO-00005,Delta_Logistics,2022-09-08,2022-09-20,Raw Materials,Delivered,1180,64.07,60.53,171.0,...,False,False,False,12.0,71425.40,4177.20,5.53,14.49,10.0,0.0


In [2]:
df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Delivery_Date'] = pd.to_datetime(df['Delivery_Date'])

In [3]:
conn = sqlite3.connect('procurement.db')
df.to_sql('purchase_orders', conn, if_exists='replace', index=False)

test = pd.read_sql_query("SELECT COUNT(*) as liczba_wierszy FROM purchase_orders", conn)
print(test)

   liczba_wierszy
0             776


In [8]:
df = df.rename(columns={
    'Savings_Rate_%': 'Savings_Rate_Pct',
    'Defect_Rate_%': 'Defect_Rate_Pct'
})

# Trzeba przeładować dane do bazy z nową nazwą kolumny
df.to_sql('purchase_orders', conn, if_exists='replace', index=False)

776

In [10]:
query_supplier_kpi = """
SELECT
    Supplier,
    COUNT(*) AS total_orders,
    ROUND(SUM(Order_Value), 2) AS total_spend,
    ROUND(AVG(Savings_Rate_Pct), 2) AS avg_savings_rate,
    ROUND(AVG(Defect_Rate_Pct), 2) AS avg_defect_rate,
    ROUND(
        SUM(CASE WHEN Is_On_Time = 1 THEN 1.0 ELSE 0.0 END)
        / SUM(CASE WHEN Is_On_Time IS NOT NULL THEN 1.0 ELSE 0.0 END),
    2) AS on_time_rate,
    ROUND(SUM(CASE WHEN Compliance = 'Yes' THEN 1.0 ELSE 0.0 END) / COUNT(*), 2) AS compliance_rate
FROM purchase_orders
GROUP BY Supplier
ORDER BY total_spend DESC;
"""
result = pd.read_sql_query(query_supplier_kpi, conn)
print(result)

          Supplier  total_orders  total_spend  avg_savings_rate  \
0    Beta_Supplies           156   9858665.90              7.83   
1    Epsilon_Group           166   9851156.06              8.04   
2  Delta_Logistics           171   9236240.47              7.81   
3         Gamma_Co           143   8587921.71              7.98   
4        Alpha_Inc           140   7825912.85              8.17   

   avg_defect_rate  on_time_rate  compliance_rate  
0             9.78          0.48             0.76  
1             3.07          0.52             0.98  
2            14.63          0.51             0.61  
3             5.02          0.57             0.86  
4             2.37          0.53             0.94  


In [11]:
query_supplier_ranking = """
WITH supplier_kpi AS (
    SELECT
        Supplier,
        COUNT(*) AS total_orders,
        ROUND(SUM(Order_Value), 2) AS total_spend,
        ROUND(AVG(Savings_Rate_Pct), 2) AS avg_savings_rate,
        ROUND(AVG(Defect_Rate_Pct), 2) AS avg_defect_rate,
        ROUND(SUM(CASE WHEN Compliance = 'Yes' THEN 1.0 ELSE 0.0 END) / COUNT(*), 2) AS compliance_rate
    FROM purchase_orders
    GROUP BY Supplier
)
SELECT
    *,
    RANK() OVER (ORDER BY avg_defect_rate ASC) AS quality_rank,
    RANK() OVER (ORDER BY avg_savings_rate DESC) AS savings_rank,
    RANK() OVER (ORDER BY compliance_rate DESC) AS compliance_rank
FROM supplier_kpi
ORDER BY total_spend DESC;
"""
result_ranking = pd.read_sql_query(query_supplier_ranking, conn)
print(result_ranking)

          Supplier  total_orders  total_spend  avg_savings_rate  \
0    Beta_Supplies           156   9858665.90              7.83   
1    Epsilon_Group           166   9851156.06              8.04   
2  Delta_Logistics           171   9236240.47              7.81   
3         Gamma_Co           143   8587921.71              7.98   
4        Alpha_Inc           140   7825912.85              8.17   

   avg_defect_rate  compliance_rate  quality_rank  savings_rank  \
0             9.78             0.76             4             4   
1             3.07             0.98             2             2   
2            14.63             0.61             5             5   
3             5.02             0.86             3             3   
4             2.37             0.94             1             1   

   compliance_rank  
0                4  
1                1  
2                5  
3                3  
4                2  


In [12]:
query_category_analysis = """
SELECT
    Item_Category,
    COUNT(*) AS total_orders,
    ROUND(SUM(Order_Value), 2) AS total_spend,
    ROUND(AVG(Savings_Rate_Pct), 2) AS avg_savings_rate,
    ROUND(AVG(Defect_Rate_Pct), 2) AS avg_defect_rate,
    ROUND(SUM(CASE WHEN Compliance = 'Yes' THEN 1.0 ELSE 0.0 END) / COUNT(*), 2) AS compliance_rate,
    ROUND(SUM(CASE WHEN Order_Status = 'Cancelled' THEN 1.0 ELSE 0.0 END) / COUNT(*), 2) AS cancellation_rate
FROM purchase_orders
GROUP BY Item_Category
ORDER BY total_spend DESC;
"""
result_category = pd.read_sql_query(query_category_analysis, conn)
print(result_category)

     Item_Category  total_orders  total_spend  avg_savings_rate  \
0              MRO           164  10126608.86              8.21   
1  Office Supplies           173   9993783.63              7.62   
2      Electronics           152   8642550.60              7.72   
3    Raw Materials           139   8471241.42              7.84   
4        Packaging           148   8125712.48              8.44   

   avg_defect_rate  compliance_rate  cancellation_rate  
0             6.40             0.82               0.05  
1             7.88             0.83               0.09  
2             6.90             0.86               0.11  
3             7.58             0.81               0.08  
4             6.28             0.80               0.08  


In [13]:
query_supplier_category = """
SELECT
    Supplier,
    Item_Category,
    COUNT(*) AS total_orders,
    ROUND(AVG(Defect_Rate_Pct), 2) AS avg_defect_rate,
    ROUND(SUM(CASE WHEN Order_Status = 'Cancelled' THEN 1.0 ELSE 0.0 END) / COUNT(*), 2) AS cancellation_rate
FROM purchase_orders
WHERE Item_Category = 'Electronics'
GROUP BY Supplier, Item_Category
ORDER BY cancellation_rate DESC;
"""
result_electronics = pd.read_sql_query(query_supplier_category, conn)
print(result_electronics)

          Supplier Item_Category  total_orders  avg_defect_rate  \
0    Beta_Supplies   Electronics            32             9.34   
1        Alpha_Inc   Electronics            22             2.35   
2  Delta_Logistics   Electronics            33            13.95   
3    Epsilon_Group   Electronics            36             3.51   
4         Gamma_Co   Electronics            29             5.12   

   cancellation_rate  
0               0.19  
1               0.09  
2               0.09  
3               0.08  
4               0.07  
